In [1]:
import sys

!{sys.executable} -m pip install tqdm netCDF4 pyarrow fastparquet


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# Step 1: Import Required Libraries
# ============================================================

import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import netCDF4 as nc

from tqdm import tqdm

In [3]:
# ============================================================
# Step 2: Project Configuration
# ============================================================

# Root project directory (directory where this notebook is located)
PROJECT_DIR = Path.cwd()

# ============================================================
# ISRO DATASET
# ============================================================

ISRO_DIR = PROJECT_DIR / "ISRO_DATASET"

ISRO_PARQUET_DIR = ISRO_DIR / "daily_parquet"

SCALER_JSON = ISRO_DIR / "lc_counts_scaler_params.json"


# ============================================================
# NASA DATASET
# ============================================================

NASA_DIR = PROJECT_DIR / "NASA_DATASET"

NASA_NC_DIR = NASA_DIR / "daily_nc"


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = PROJECT_DIR / "MERGED_DATASET"

# Create output folder if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Display Paths
# ============================================================

print("=" * 60)
print("PROJECT DIRECTORY")
print(PROJECT_DIR)

print("\nISRO PARQUET DIRECTORY")
print(ISRO_PARQUET_DIR)

print("\nNASA NC DIRECTORY")
print(NASA_NC_DIR)

print("\nSCALER JSON")
print(SCALER_JSON)

print("\nOUTPUT DIRECTORY")
print(OUTPUT_DIR)
print("=" * 60)

PROJECT DIRECTORY
c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest

ISRO PARQUET DIRECTORY
c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\ISRO_DATASET\daily_parquet

NASA NC DIRECTORY
c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\NASA_DATASET\daily_nc

SCALER JSON
c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\ISRO_DATASET\lc_counts_scaler_params.json

OUTPUT DIRECTORY
c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\MERGED_DATASET


In [4]:
# ============================================================
# Step 3: Load Global Scaling Parameters
# ============================================================

with open(SCALER_JSON, "r") as f:
    scaler_params = json.load(f)

global_mean = scaler_params["lc_counts_log_mean"]
global_std = scaler_params["lc_counts_log_std"]

print("=" * 60)
print("Scaling Parameters Loaded Successfully")
print("=" * 60)
print(f"Transform   : {scaler_params['transform']}")
print(f"Files Used  : {scaler_params['files_used']}")
print(f"Global Mean : {global_mean}")
print(f"Global Std  : {global_std}")
print("=" * 60)

Scaling Parameters Loaded Successfully
Transform   : log1p_then_zscore
Files Used  : 748
Global Mean : 3.5101029872894287
Global Std  : 1.3320047855377197


In [5]:
# ============================================================
# Step 4 : ISRO Preprocessing Functions
# ============================================================

# -------------------------------
# Required column definitions
# -------------------------------

CHANNEL_COLS = [f"ch_{i:03d}" for i in range(340)]

META_COLS = [
    "unix_time",
    "lc_counts",
    "valid_counts",
    "hardness_ratio",
    "in_gti"
]


# -------------------------------
# SoLEXS preprocessing
# -------------------------------

def preprocess_solexs(df, verbose=False):
    """
    Clean a single SoLEXS dataframe.
    """

    n_raw = len(df)

    # Step 1 : Keep required columns
    available_ch = [c for c in CHANNEL_COLS if c in df.columns]
    available_meta = [c for c in META_COLS if c in df.columns]

    df_main = df[available_meta + available_ch].copy()

    # Step 2 : Keep only valid GTI rows
    if "in_gti" in df_main.columns:
        df_main = df_main[df_main["in_gti"] == 1].copy()
        df_main.drop(columns=["in_gti"], inplace=True)

    n_gti = len(df_main)

    # Step 3 : Fill missing values
    df_main[available_ch] = df_main[available_ch].fillna(0)

    if "hardness_ratio" in df_main.columns:
        df_main["hardness_ratio"] = df_main["hardness_ratio"].fillna(
            df_main["hardness_ratio"].median()
        )

    key_cols = [
        c for c in
        ["unix_time", "lc_counts", "valid_counts"]
        if c in df_main.columns
    ]

    df_main = df_main.dropna(subset=key_cols)

    n_clean = len(df_main)

    # Step 4 : Check negative counts
    if "lc_counts" in df_main.columns:

        neg_lc = (df_main["lc_counts"] < 0).sum()

        if neg_lc > 0 and verbose:
            print(f"WARNING : {neg_lc} negative lc_counts found.")

    # Step 5 : Downcast channels
    df_main[available_ch] = df_main[available_ch].astype(np.float32)

    if verbose:

        print(f"Rows : {n_raw} -> {n_gti} -> {n_clean}")
        print(f"Columns : {df_main.shape[1]}")

    return df_main


# -------------------------------
# Global scaling
# -------------------------------

def apply_lc_scaling(df_main, global_mean, global_std):

    if "lc_counts" not in df_main.columns:
        return df_main

    df_main["lc_counts_log"] = np.log1p(df_main["lc_counts"])

    df_main["lc_counts_scaled"] = (
        (df_main["lc_counts_log"] - global_mean)
        / global_std
    )

    # Defragment dataframe
    return df_main.copy()


# -------------------------------
# Complete preprocessing pipeline
# -------------------------------

def preprocess_isro(parquet_file, verbose=False):
    """
    Complete preprocessing for one ISRO parquet file.
    """

    # Read parquet
    df = pd.read_parquet(parquet_file)

    # Basic preprocessing
    df = preprocess_solexs(df, verbose)

    # Global scaling
    df = apply_lc_scaling(
        df,
        global_mean,
        global_std
    )

    return df


print("ISRO preprocessing functions loaded successfully.")

ISRO preprocessing functions loaded successfully.


In [6]:
# ============================================================
# Step 5 : GOES (NASA) Preprocessing Function
# ============================================================

def preprocess_goes(nc_file, isro_time):
    """
    Preprocess one GOES NetCDF file.

    Parameters
    ----------
    nc_file : str or Path
        Path to GOES .nc file

    isro_time : pandas.Series
        unix_time column from corresponding ISRO dataframe

    Returns
    -------
    pandas.DataFrame
    """

    # ========================================================
    # Step 1 : Read NetCDF
    # ========================================================

    ds = nc.Dataset(nc_file)

    df = pd.DataFrame({
        "time": ds.variables["time"][:],
        "xrsb_flux": ds.variables["xrsb_flux"][:],
        "status": ds.variables["status"][:],
        "background_flux": ds.variables["background_flux"][:],
        "flare_class": ds.variables["flare_class"][:],
        "integrated_flux": ds.variables["integrated_flux"][:],
        "flare_id": ds.variables["flare_id"][:],
        "sequential_flare_num": ds.variables["sequential_flare_num"][:]
    })

    ds.close()

    # ========================================================
    # Step 2 : Remove POST_EVENT
    # ========================================================

    df = df[df["status"] != "POST_EVENT"].reset_index(drop=True)

    # ========================================================
    # Step 3 : Convert GOES time to Unix timestamp
    # ========================================================

    # Convert NetCDF time to datetime
    base_time = pd.Timestamp("2000-01-01 12:00:00", tz="UTC")

    df["time"] = (
        base_time +
        pd.to_timedelta(df["time"], unit="s")
    )

    # Convert datetime to Unix timestamp (seconds)
    df["time"] = (
        df["time"].astype("int64") // 10**9
    ).astype("int64")

    # ========================================================
    # Step 4 : Keep only observation day
    # ========================================================

    dt = pd.to_datetime(df["time"], unit="s")

    observation_date = dt.dt.date.max()

    df = df[dt.dt.date == observation_date].copy()

    # ========================================================
    # Step 5 : Expand using ISRO timeline
    # ========================================================

    full_time_df = pd.DataFrame({
        "time": isro_time.astype(np.int64)
    })

    df = full_time_df.merge(
        df,
        on="time",
        how="left"
    )

    # ========================================================
    # Step 6 : Average background flux
    # ========================================================

    avg_background_flux = df["background_flux"].mean(skipna=True)

    # ========================================================
    # Step 7 : Fill Region 3
    # ========================================================

    end_indices = df.index[df["status"] == "EVENT_END"].tolist()
    start_indices = df.index[df["status"] == "EVENT_START"].tolist()

    for end_idx in end_indices:

        next_start = next(
            (idx for idx in start_indices if idx > end_idx),
            None
        )

        if next_start is None:
            region = slice(end_idx + 1, len(df))
        else:
            region = slice(end_idx + 1, next_start)

        mask = df.loc[region, "xrsb_flux"].isna()

        df.loc[region, "xrsb_flux"] = (
            df.loc[region, "xrsb_flux"]
              .where(~mask, avg_background_flux)
        )

    # ========================================================
    # Step 8 : Keep required columns
    # ========================================================

    df = df[
        [
            "time",
            "xrsb_flux",
            "status",
            "flare_class"
        ]
    ].copy()

    # Rename for merging
    df.rename(
        columns={"time": "unix_time"},
        inplace=True
    )

    return df


print("GOES preprocessing function loaded successfully.")

GOES preprocessing function loaded successfully.


In [7]:
# # ============================================================
# # Step 6.2 : Merge ISRO + GOES
# # ============================================================

# merged_df = pd.merge(
#     df_isro,
#     df_goes,
#     on="unix_time",
#     how="left",
#     validate="one_to_one"
# )

# print("Merge successful!")

# print("\nShape :", merged_df.shape)

# print("\nLast 4 columns:")
# print(merged_df.columns[-4:].tolist())

In [8]:
# # ============================================================
# # Keep only required columns
# # ============================================================

# channel_cols = [f"ch_{i:03d}" for i in range(340)]

# columns_to_keep = (
#     [
#         "unix_time",
#         "lc_counts_scaled",
#         "hardness_ratio",
#     ]
#     + channel_cols
#     + [
#         "xrsb_flux",
#         "status",
#         "flare_class",
#     ]
# )

# merged_df = merged_df[columns_to_keep].copy()

# print("Final Shape :", merged_df.shape)

# print("\nFinal Columns:")
# print(merged_df.columns.tolist())

In [9]:
print(OUTPUT_DIR)

c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\MERGED_DATASET


In [10]:
# ============================================================
# PROCESS ALL FILES
# ============================================================

import re

# ------------------------------------------------------------
# Build file dictionaries using date as key
# ------------------------------------------------------------

isro_files = {}

for f in ISRO_PARQUET_DIR.glob("*.parquet"):
    m = re.search(r"(\d{8})", f.name)
    if m:
        isro_files[m.group(1)] = f

goes_files = {}

for f in NASA_NC_DIR.glob("*.nc"):
    m = re.search(r"d(\d{8})", f.name)
    if m:
        goes_files[m.group(1)] = f

# ------------------------------------------------------------
# Common dates
# ------------------------------------------------------------

common_dates = sorted(set(isro_files.keys()) & set(goes_files.keys()))

print("=" * 60)
print(f"ISRO files : {len(isro_files)}")
print(f"GOES files : {len(goes_files)}")
print(f"Matched    : {len(common_dates)}")
print("=" * 60)
print("First 5 matched dates:")
print(common_dates[:5])
# ------------------------------------------------------------
# Columns to keep
# ------------------------------------------------------------

channel_cols = [f"ch_{i:03d}" for i in range(340)]

columns_to_keep = (
    [
        "unix_time",
        "lc_counts_scaled",
        "hardness_ratio",
    ]
    + channel_cols
    + [
        "xrsb_flux",
        "status",
        "flare_class",
    ]
)

# ------------------------------------------------------------
# Processing loop
# ------------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
success = 0
failed = []

for date in tqdm(common_dates):

    try:

        # ----------------------------------------------------
        # ISRO
        # ----------------------------------------------------

        df_isro = pd.read_parquet(isro_files[date])

        df_isro = preprocess_solexs(df_isro)

        df_isro = apply_lc_scaling(
            df_isro,
            global_mean,
            global_std
        )

        # ----------------------------------------------------
        # GOES
        # ----------------------------------------------------

        df_goes = preprocess_goes(
            goes_files[date],
            df_isro["unix_time"]
        )

        # ----------------------------------------------------
        # Merge
        # ----------------------------------------------------

        merged_df = pd.merge(
            df_isro,
            df_goes,
            on="unix_time",
            how="left",
            validate="one_to_one"
        )

        merged_df = merged_df[columns_to_keep].copy()

        merged_df["unix_time"] = merged_df["unix_time"].astype(np.int64)

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        output_file = OUTPUT_DIR / f"merged_{date}.parquet"

        merged_df.to_parquet(
            output_file,
            index=False
        )

        success += 1

    except Exception as e:

        failed.append((date, str(e)))
        print(f"\nFAILED : {date}")
        print(e)
        
# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PROCESS COMPLETE")
print("=" * 60)

print(f"Successful : {success}")
print(f"Failed     : {len(failed)}")

if failed:

    print("\nFailed files:")

    for date, err in failed:
        print(f"{date} -> {err}")

ISRO files : 733
GOES files : 733
Matched    : 733
First 5 matched dates:
['20240201', '20240202', '20240203', '20240204', '20240205']


  0%|          | 0/733 [00:00<?, ?it/s]C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_main["lc_counts_log"] = np.log1p(df_main["lc_counts"])
C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_main["lc_counts_scaled"] = (
  0%|          | 1/733 [00:02<36:03,  2.96s/it]C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usua


FAILED : 20240515
Merge keys are not unique in right dataset; not a one-to-one merge


C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_main["lc_counts_log"] = np.log1p(df_main["lc_counts"])
C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_main["lc_counts_scaled"] = (
 13%|█▎        | 97/733 [02:57<16:37,  1.57s/it]C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inser


FAILED : 20240528
Merge keys are not unique in right dataset; not a one-to-one merge


...


FAILED : 20250221
Merge keys are not unique in right dataset; not a one-to-one merge


C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_main["lc_counts_log"] = np.log1p(df_main["lc_counts"])
C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_main["lc_counts_scaled"] = (
 45%|████▍     | 327/733 [09:45<11:09,  1.65s/it]C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse


FAILED : 20260405
Merge keys are not unique in right dataset; not a one-to-one merge


C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_main["lc_counts_log"] = np.log1p(df_main["lc_counts"])
C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_main["lc_counts_scaled"] = (
 91%|█████████ | 668/733 [20:03<01:36,  1.48s/it]C:\Users\harsh\AppData\Local\Temp\ipykernel_4568\2015887828.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inse


PROCESS COMPLETE
Successful : 728
Failed     : 5

Failed files:
20240515 -> Merge keys are not unique in right dataset; not a one-to-one merge
20240528 -> Merge keys are not unique in right dataset; not a one-to-one merge
20241213 -> Merge keys are not unique in right dataset; not a one-to-one merge
20250221 -> Merge keys are not unique in right dataset; not a one-to-one merge
20260405 -> Merge keys are not unique in right dataset; not a one-to-one merge


In [11]:
from pathlib import Path
import pandas as pd

# ============================================================
# Convert first 5 merged parquet files to CSV
# ============================================================

PARQUET_DIR = OUTPUT_DIR                  # MERGED_DATASET
CSV_DIR = OUTPUT_DIR / "sample_csv"

CSV_DIR.mkdir(exist_ok=True)

parquet_files = sorted(PARQUET_DIR.glob("*.parquet"))

print(f"Found {len(parquet_files)} parquet files.\n")

for file in parquet_files[:5]:

    df = pd.read_parquet(file)

    csv_name = file.stem + ".csv"

    df.to_csv(
        CSV_DIR / csv_name,
        index=False
    )

    print(f"Converted: {file.name} -> {csv_name}")

print("\nDone!")
print(f"CSV files saved in:\n{CSV_DIR}")

Found 728 parquet files.

Converted: merged_20240201.parquet -> merged_20240201.csv
Converted: merged_20240202.parquet -> merged_20240202.csv
Converted: merged_20240203.parquet -> merged_20240203.csv
Converted: merged_20240204.parquet -> merged_20240204.csv
Converted: merged_20240205.parquet -> merged_20240205.csv

Done!
CSV files saved in:
c:\Users\harsh\OneDrive\Desktop\AIML\DL\ISRO_HACKATHON\dataset\SolexCopyTest\MERGED_DATASET\sample_csv
